In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/candidates.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (73462, 15)


,Unnamed: 0,Age,Accessibility,EdLevel,Employment,Gender,MentalHealth,MainBranch,YearsCode,YearsCodePro,Country,PreviousSalary,HaveWorkedWith,ComputerSkills,Employed
0,0,<35,No,Master,1,Man,No,Dev,7,4,Sweden,51552.0,C++;Python;Git;PostgreSQL,4,0
1,1,<35,No,Undergraduate,1,Man,No,Dev,12,5,Spain,46482.0,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,12,1
2,2,<35,No,Master,1,Man,No,Dev,15,6,Germany,77290.0,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,7,0
3,3,<35,No,Undergraduate,1,Man,No,Dev,9,6,Canada,46135.0,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,13,0
4,4,>35,No,PhD,0,Man,No,NotDev,40,30,Singapore,160932.0,C++;Python,2,0


## Stage A — Understand & Set the Bar

**Data source note:** This task requires real production marketplace data (matching/ranking events), which was not available. As an honest substitute, the Stack Overflow Developer Survey dataset (73,463 real developer profiles, Kaggle) is used — real data, not synthetic, but candidate-side only. Job-side requirements below are authored for this exercise, not sourced from real job postings.

**What "good" means here:** I can say objectively whether a candidate-ranking approach improved in terms the business cares about — specifically, whether it surfaces more genuinely skill-relevant, employable candidates near the top of a ranked list than a naive baseline does.

In [3]:
job_profiles = {
    "backend_engineer": {"Python", "SQL", "Git", "PostgreSQL", "Docker", "AWS"},
    "frontend_engineer": {"JavaScript", "HTML/CSS", "React.js", "TypeScript", "Git"},
    "data_analyst": {"SQL", "Python", "Excel", "Git", "PostgreSQL"},
}

In [4]:
def relevance_score(candidate_skills_str, required_skills):
    if pd.isna(candidate_skills_str):
        return 0.0
    candidate_skills = set(candidate_skills_str.split(";"))
    overlap = candidate_skills & required_skills
    return len(overlap) / len(required_skills)

df["relevance_backend"] = df["HaveWorkedWith"].apply(
    lambda s: relevance_score(s, job_profiles["backend_engineer"])
)

df[["HaveWorkedWith", "relevance_backend"]].head(10)

,HaveWorkedWith,relevance_backend
0,C++;Python;Git;PostgreSQL,0.500000
1,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,0.666667
2,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,0.166667
3,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,0.666667
4,C++;Python,0.166667
5,JavaScript;Python;Docker;Git;MySQL,0.500000
6,C++;HTML/CSS;Java;JavaScript;Kotlin;Node.js;Ty...,0.666667
7,C++;Python;Docker;Git,0.500000
8,Python;Git;PostgreSQL,0.500000
9,Delphi;Java;SQL;Docker;Git;PostgreSQL,0.666667


In [5]:
# Create a binary proxy flag for employability outcome
df["is_employed"] = df["Employed"].apply(
    lambda x: 1 if x == 1 or str(x).strip().lower() in ["true", "1", "employed full-time"] else 0
)

print("Employment outcome distribution:")
print(df["is_employed"].value_counts(normalize=True))

Employment outcome distribution:
is_employed
1    0.536223
0    0.463777
Name: proportion, dtype: float64


In [6]:
# 1. Naive Baseline: Unsorted default candidate list
baseline_top20 = df.head(20)

# 2. Ranking Model: Sorted by skill relevance, then employment status
ranked_top20 = df.sort_values(
    by=["relevance_backend", "is_employed"], ascending=[False, False]
).head(20)

# Evaluate Mean Relevance @ 20
baseline_mrr = baseline_top20["relevance_backend"].mean()
ranked_mrr = ranked_top20["relevance_backend"].mean()

print(f"--- EVALUATION RESULTS ---")
print(f"Naive Baseline Mean Relevance @ 20:  {baseline_mrr:.4f}")
print(f"Relevance Ranker Mean Relevance @ 20: {ranked_mrr:.4f}")
print(f"Relative Improvement:               {((ranked_mrr - baseline_mrr) / (baseline_mrr + 1e-9)) * 100:.2f}%")

--- EVALUATION RESULTS ---
Naive Baseline Mean Relevance @ 20:  0.4917
Relevance Ranker Mean Relevance @ 20: 1.0000
Relative Improvement:               103.39%


## Stage D — Metric Defense & Gaps

* **Relevance Metric:** Evaluated as Jaccard-style skill coverage against required job requirements.
* **Hire Rate Metric:** Approximated using the binary employment status signal (`is_employed`).
* **Shortlist Rate & Time-to-Hire Gaps:** Not computable from candidate profile data. Fully measuring these metrics requires ATS/funnel event timestamps (recruiter views, shortlist clicks, offer acceptances).

## Day 11 — Matching & Ranking Evaluation Framework
* **Dataset:** Stack Overflow Developer Survey (73,463 candidates; external proxy for live marketplace data).
* **Metrics Built:**
  * **Mean Relevance @ 20:** Jaccard skill overlap against backend profile (Baseline: 0.4917 vs. Ranker: 1.0000 | +103.39%).
  * **Employability Signal:** Binary flag derived from candidate employment status (`is_employed`).
* **System Gaps Noted:** Shortlist rate and Time-to-hire require upstream ATS/recruiter timestamp logs not present in candidate profile data.